In [1]:
import pandas as pd
import numpy as np
import re

crosswalk = pd.read_excel('../data/raw/TRACT_ZIP_122022.xlsx')

def decode_excel_xml(s):
    return re.sub(r'_x([0-9A-Fa-f]{4})_', 
                  lambda m: chr(int(m.group(1), 16)), str(s))

crosswalk['TRACT'] = crosswalk['TRACT'].apply(decode_excel_xml)
crosswalk['ZIP']   = crosswalk['ZIP'].apply(decode_excel_xml)


print("after decode TRACT:")
print(crosswalk['TRACT'].head(5).tolist())
print()
print("after decode ZIP:")
print(crosswalk['ZIP'].head(5).tolist())

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


after decode TRACT:
['01001020100', '01001020200', '01001020300', '01001020400', '01001020400']

after decode ZIP:
['36067', '36067', '36067', '36067', '36066']


In [2]:
noise = pd.read_csv(
    '../data/processed/tract/tract_noise.csv'
)

print(noise.head())
print(noise.shape)

     tract_id  tract_name  noise_mean_db  noise_max_db  noise_min_db
0  6073008331       83.31      56.328147     81.841232     45.003738
1  6073008336       83.36      52.557514     60.831444     45.011963
2  6073008337       83.37      58.447012     81.628632     45.016068
3  6073011601      116.01      51.534390     59.524536     45.223137
4  6073011602      116.02      55.575460     82.591164     45.002365
(737, 5)


In [3]:

crosswalk_sd = crosswalk[crosswalk['TRACT'].str.contains('6073')].copy()
crosswalk_sd = crosswalk_sd.rename(columns={'TRACT': 'tract_id', 'ZIP': 'zcta'})

print("San Diego num:", len(crosswalk_sd))
print("TRACT sample:", crosswalk_sd['tract_id'].head(3).tolist())
print("ZIP sample:", crosswalk_sd['zcta'].head(3).tolist())

San Diego num: 1196
TRACT sample: ['06073000100', '06073000100', '06073000201']
ZIP sample: ['92103', '92110', '92103']


In [4]:

crosswalk_sd['tract_id'] = crosswalk_sd['tract_id'].astype('int64')


noise_mapped = pd.merge(noise, crosswalk_sd[['tract_id', 'zcta', 'RES_RATIO']], 
                         on='tract_id', how='left')

print("merge num :", len(noise_mapped))
print("have't match tract:", noise_mapped['zcta'].isna().sum())
print()
print(noise_mapped.head(3).to_string())

merge num : 1073
have't match tract: 203

     tract_id  tract_name  noise_mean_db  noise_max_db  noise_min_db   zcta  RES_RATIO
0  6073008331       83.31      56.328147     81.841232     45.003738  92130        1.0
1  6073008336       83.36      52.557514     60.831444     45.011963  92129        1.0
2  6073008337       83.37      58.447012     81.628632     45.016068  92129        1.0


In [5]:
def weighted_noise(g):
    total_weight = g['RES_RATIO'].sum()
    if total_weight == 0:
        return pd.Series({
            'noise_mean_db': g['noise_mean_db'].mean(),
            'noise_max_db':  g['noise_max_db'].mean(),
            'noise_min_db':  g['noise_min_db'].mean(),
        })
    return pd.Series({
        'noise_mean_db': np.average(g['noise_mean_db'], weights=g['RES_RATIO']),
        'noise_max_db':  np.average(g['noise_max_db'],  weights=g['RES_RATIO']),
        'noise_min_db':  np.average(g['noise_min_db'],  weights=g['RES_RATIO']),
    })

noise_zcta = (
    noise_mapped.dropna(subset=['zcta']) 
    .groupby('zcta')
    .apply(weighted_noise)
    .reset_index()
)

print("ZCTA:", len(noise_zcta))
print("zcta sample:", noise_zcta['zcta'].head(5).tolist())
print(noise_zcta.head(5).to_string())
noise_zcta.to_csv('../data/processed/zcta/zcta_noise.csv', index=False)
print("✅ ")

ZCTA: 154
zcta sample: ['91901', '91902', '91903', '91910', '91911']
    zcta  noise_mean_db  noise_max_db  noise_min_db
0  91901      57.659220     80.190165     45.006918
1  91902      54.664245     71.750261     45.571662
2  91903      58.401917     80.374268     45.002365
3  91910      55.273382     74.431252     45.067791
4  91911      55.898695     74.625658     45.040166
✅ 


In [6]:
# read all
socioeconomic = pd.read_csv('../data/processed/zcta/zcta_demographics_selected.csv')
demographics  = pd.read_csv('../data/processed/zcta/zcta_demographics.csv')
housing       = pd.read_csv('../data/processed/zcta/zcta_housing.csv')
housing_sd    = housing[housing['county_name'] == 'San Diego County'].rename(columns={'zip_code': 'zcta'})

noise_zcta['zcta']    = noise_zcta['zcta'].astype(str).str.zfill(5)
socioeconomic['zcta'] = socioeconomic['zcta'].astype(str).str.zfill(5)
demographics['zcta']  = demographics['zcta'].astype(str).str.zfill(5)
housing_sd['zcta']    = housing_sd['zcta'].astype(str).str.zfill(5)

# Merge
master = noise_zcta.copy()
master = pd.merge(master, socioeconomic, on='zcta', how='left')
master = pd.merge(master, demographics,  on='zcta', how='left')
master = pd.merge(master, housing_sd[['zcta', 'home_value']], on='zcta', how='left')

master['poverty_rate'] = master['poverty_rate_x'].combine_first(master['poverty_rate_y'])
master = master.drop(columns=['poverty_rate_x', 'poverty_rate_y'])

print("Shape:", master.shape)
print("\nmissing:")
print(master.isnull().sum())

# 保存
master.to_csv('../data/processed/master_dataset.csv', index=False)
print("\ncomplete")

Shape: (154, 17)

missing:
zcta                        0
noise_mean_db               1
noise_max_db                1
noise_min_db                1
median_household_income    62
unemployment_rate          51
population                 63
pnhwhite                   63
pnhblack                   63
phispanic                  63
pforeign_born              63
punemployed                63
affluence                  63
disadvantage               63
median_family_income       63
home_value                 65
poverty_rate               60
dtype: int64

complete
